# MNIST MLP3: AdamW versus AdaptiveSpectralGuard

This notebook tests the second optimizer implementation in `rg_optimizers`.

The optimizer is **not** the original every-step trace-log filter. It combines:

1. layer-specific cadence, gain, and norm caps;
2. a WeightWatcher-driven off/weak/strong controller with hysteresis;
3. an ECS-confidence gate based on PL/ERG overlap, support stability, and `ERG_gap`;
4. one-sided retained trace-log branch protection;
5. a trace-log-preserving shell-$\beta_E$ shape correction;
6. a first-order task-loss safeguard;
7. task-conflict feedback that automatically throttles harmful layers.

For the two spectral channels,

$$
\Delta W_{\mathrm{guard}}
=
\Delta W_T+\Delta W_\beta ,
$$

where $\Delta W_T$ removes only contracting trace-log drift and
$\Delta W_\beta$ moves reliable shell imbalance toward $\beta_E=0$ after
orthogonalization against the trace-log normal.

The task safeguard enforces

$$
\left\langle \nabla_W L,\Delta W_{\mathrm{guard}}\right\rangle_F
\leq 0
$$

by default, so the correction cannot increase the current minibatch loss to
first order.

The main run is 30 epochs. FC1 is corrected relatively frequently, FC2 is
corrected conservatively, and FC3 is disabled. Optional FC1-only and FC2-only
ablation presets are provided at the end.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import weightwatcher as ww
from IPython.display import display

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "adaptive_spectral_guard").is_dir():
        ROOT = candidate
        break
    nested = candidate / "optimizers" / "adaptive_spectral_guard"
    if (nested / "adaptive_spectral_guard").is_dir():
        ROOT = nested
        break

if ROOT is None:
    raise RuntimeError(
        "Open this notebook from a clone of rg_optimizers."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from adaptive_spectral_guard import (
    GuardConfig,
    MNISTGuardExperimentConfig,
    preset_policies,
    run_mnist_guard_comparison,
)
from adaptive_spectral_guard.plotting import (
    plot_controller,
    plot_corrections,
    plot_matched_convergence,
    plot_performance,
    plot_weightwatcher,
)

print("Package root:", ROOT)
print("Torch:", torch.__version__)
print("WeightWatcher:", getattr(ww, "__version__", "unknown"))

## Configure the main adaptive run

The default policy is deliberately asymmetric:

- **FC1:** cadence 2, stronger volume protection, small $\beta_E$ shape channel;
- **FC2:** cadence 10, smaller gain and tighter caps;
- **FC3:** disabled.

The slow controller can still turn FC1 or FC2 completely off when its
WeightWatcher state is safely above the boundary or its ECS confidence is
poor.

In [ ]:
PRESET = "adaptive"

experiment_config = MNISTGuardExperimentConfig(
    seed=1337,
    epochs=30,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-2,
    grad_clip_norm=1.0,
    ww_min_evals=10,
    ww_max_evals=None,
    n_shells=5,
    min_beta_retained=20,
    min_beta_decades=0.50,
    train_eval_max_batches=None,
)

guard_config = GuardConfig(
    policies=preset_policies(PRESET),
)

RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = (
    ROOT
    / f"results_adaptive_spectral_guard_{PRESET}_30epochs_{RUN_STAMP}"
)

display(
    pd.DataFrame(
        {
            name: vars(policy)
            for name, policy in guard_config.policies.items()
        }
    ).T
)
print("Results:", OUTPUT_DIR.resolve())

## Run AdamW and AdamW + AdaptiveSpectralGuard

A complete report is printed after every epoch:

- train/test loss and accuracy;
- WeightWatcher $\alpha$, `ERG_gap`, midpoint rank, and $\beta_E$;
- off/weak/strong controller state and effective gain;
- correction magnitude and firing rate;
- attempted and post-safeguard task conflict.

Create the printed `STOP_AFTER_CURRENT_EPOCH` file from another terminal to
stop cleanly after the current epoch. A `KeyboardInterrupt` also preserves all
completed epochs.

In [ ]:
result = run_mnist_guard_comparison(
    experiment_config,
    guard_config,
    data_dir=ROOT / "data",
    output_dir=OUTPUT_DIR,
    progress=True,
)

print("Completed through epoch:", int(result.performance["epoch"].max()))
print("Saved to:", result.output_dir.resolve())

## Performance and WeightWatcher trajectories

In [ ]:
plot_performance(result.performance)

plot_weightwatcher(
    result.weightwatcher,
    "alpha",
    reference=2.0,
    title=r"All layers: WeightWatcher $\alpha$",
    ylabel=r"WeightWatcher $\alpha$",
)

plot_weightwatcher(
    result.weightwatcher,
    "ERG_gap",
    reference=0.0,
    title="All layers: WeightWatcher ERG gap",
    ylabel="WeightWatcher ERG gap",
)

plot_weightwatcher(
    result.weightwatcher,
    "beta_E_midpoint",
    reference=0.0,
    title=r"All layers: shell $\beta_E$",
    ylabel=r"Shell $\beta_E$",
)

## Controller, correction, and task-conflict diagnostics

A positive pre-safeguard conflict ratio means the proposed spectral correction
would oppose task descent:

$$
q_\ell
=
\frac{
\left\langle \nabla_\ell L,C_\ell\right\rangle_F
}{
\left|
\left\langle \nabla_\ell L,\Delta W_{\ell,\mathrm{AdamW}}\right\rangle_F
\right|+\varepsilon
}.
$$

The post-safeguard ratio should be non-positive up to numerical error. A high
attempted conflict rate causes the next-epoch task throttle to reduce that
layer's effective gain.

In [ ]:
display(result.controller.tail(12))
display(result.correction_summary.tail(12))

plot_controller(result.controller)
plot_corrections(result.correction_summary)

## Compare at matched convergence

Matched-epoch plots can mistake slower optimization for better
generalization. These plots compare:

$$
L_{\mathrm{test}}\ \text{versus}\ L_{\mathrm{train}},
\qquad
A_{\mathrm{test}}\ \text{versus}\ A_{\mathrm{train}},
$$

and each layer's WeightWatcher $\alpha$ versus training loss.

In [ ]:
plot_matched_convergence(
    result.performance,
    result.weightwatcher,
)

## Direct source and safety checks

In [ ]:
ok = result.weightwatcher.loc[
    result.weightwatcher["status"].eq("ok")
].copy()

assert ok["alpha_source"].astype(str).eq("WeightWatcher").all()
assert ok["ERG_gap_source"].astype(str).eq("WeightWatcher").all()

if not result.guard_steps.empty:
    post_conflict = pd.to_numeric(
        result.guard_steps["task_conflict_ratio_post"],
        errors="coerce",
    ).dropna()
    print("Maximum post-safeguard task conflict:", post_conflict.max())
    print(
        "Fraction positive after safeguard:",
        (post_conflict > 1e-6).mean(),
    )

display(
    ok[
        [
            "run",
            "epoch",
            "layer_name",
            "alpha",
            "ERG_gap",
            "beta_E_midpoint",
            "scale_balance_reliable",
        ]
    ].tail(18)
)

## Optional layer ablations

Set `RUN_ABLATIONS=True` to run separate 30-epoch experiments with:

- FC1 correction only;
- FC2 correction only;
- the full adaptive FC1+FC2 policy.

This distinguishes direct FC2 overconstraint from an indirect effect caused
by changing FC1's representation.

In [ ]:
RUN_ABLATIONS = False
ABLATION_PRESETS = ["fc1_only", "fc2_only", "adaptive"]
ablation_results = {}

if RUN_ABLATIONS:
    for preset in ABLATION_PRESETS:
        stamp = time.strftime("%Y%m%d_%H%M%S")
        ablation_output = (
            ROOT
            / f"results_adaptive_spectral_guard_ablation_{preset}_{stamp}"
        )
        print("\nRUNNING ABLATION:", preset)
        ablation_results[preset] = run_mnist_guard_comparison(
            experiment_config,
            GuardConfig(policies=preset_policies(preset)),
            data_dir=ROOT / "data",
            output_dir=ablation_output,
            progress=True,
        )

    summary_rows = []
    for preset, ablation in ablation_results.items():
        final = (
            ablation.performance
            .sort_values("epoch")
            .groupby("run")
            .tail(1)
        )
        for _, row in final.iterrows():
            summary_rows.append(
                {
                    "preset": preset,
                    "run": row["run"],
                    "epoch": row["epoch"],
                    "train_loss": row["train_loss"],
                    "train_acc": row["train_acc"],
                    "test_loss": row["test_loss"],
                    "test_acc": row["test_acc"],
                }
            )
    display(pd.DataFrame(summary_rows))

## Interpretation

The new optimizer is useful only if it does more than delay training.

Evidence for a real overfitting barrier requires lower test loss, a smaller
test-loss rebound, or better test accuracy **at matched training progress**.
If FC2 remains slow while its attempted task-conflict rate is high, the
controller should reduce its task throttle and effective gain. If the
post-safeguard conflict remains positive, that is an implementation failure.

The shell-$\beta_E$ channel is intentionally small and trace-log orthogonal.
It is present to address the residual FC1 drift below $\alpha=2$ without
turning the entire optimizer into a hard projection onto $\alpha=2$.